#Telecom Domain Read & Write Ops Assignment - Building Datalake & Lakehouse
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object

### create spark session

In [0]:
from spark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()


##1. Write SQL statements to create:
1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol
4. Using dbutils.fs.mkdirs, create folders:<br>
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):<br>
a. Volume vs DBFS/FileStore<br>
b. Why production teams prefer Volumes for regulated data<br>

###  Create Catalog/Schema/volume

In [0]:
%sql
create catalog if not exists telecom_catalog_assign;
create schema if not exists landing_zone;
create volume if not exists landing_vol;

### Create directory/ folder in two ways

1. %fs mkdirs
2. dbutils.fs.mkdirs("path")

In [0]:
%fs mkdirs /Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/

In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")

### Volume vs DBFS/FileStore

Volume - is the databricks recommended storage, because they integrates with unity catalog and provides better security and governance(centerialised access control) <br>
**Use Volumes when:**<br>
- Working with production data.<br>
- Sharing files securely.<br>
- Using Unity Catalog.<br>
- Building new Databricks projects.


DBFS /File Store - File store is a folder in Databricks file system. it is legacy storage mechanism and is not recommended for new production workloads.<br>
**Use FileStore when:**<br>
- Maintaining older notebooks.<br>
- Working with legacy code.<br>
- Storing temporary notebook assets (if required). 

##Data files to use in this usecase:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

### Load customer_csv string to volume

In [0]:
customer_csv = ''' 101,Arun,31,Chennai,PREPAID 102,Meera,45,Bangalore,POSTPAID 103,Irfan,29,Hyderabad,PREPAID 104,Raj,52,Mumbai,POSTPAID 105,,27,Delhi,PREPAID 106,Sneha,abc,Pune,PREPAID '''

rows = [line.split(",") for line in customer_csv.strip().split(" ")]

df = spark.createDataFrame(rows, ["id", "name", "age", "city", "plan"])

df.show()

df.write.mode("overwrite").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",header=True)

In [0]:

%fs ls /Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/

In [0]:
read_cust_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",header=True,inferSchema=True)
read_cust_df.show()
read_cust_df.printSchema()

### Load usage_csv string to volume

In [0]:
usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count 101\t320\t1500\t20 102\t120\t4000\t5 103\t540\t600\t52 104\t45\t200\t2 105\t0\t0\t0 '''

lines = usage_tsv.strip().split(" ")
print(lines)
columns = lines[0].split("\t")

rows = [line.split("\t") for line in lines[1:]]

usage_df = spark.createDataFrame(rows, columns)

usage_df.show()

usage_df.write.mode("overwrite").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.csv",header=True)




In [0]:
read_usage_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.csv",inferSchema=True,header=True)
read_usage_df.show()

read_usage_df.printSchema()

### Load tower_logs_region1 string to volume

In [0]:
import re
tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp 5001|101|TWR01|-80|2025-01-10 10:21:54 5004|104|TWR05|-75|2025-01-10 11:01:12 '''

parts=tower_logs_region1.strip(" ").split(" ",1)
print(parts)

columns=parts[0].split("|")
print(columns)


rows = re.findall(r'\d+\|.*?(?=\s+\d+\||$)', parts[1])
print(rows)
data=[row.split("|") for row in rows]
print(data)


tower_logs_region1_df=spark.createDataFrame(data,columns)
tower_logs_region1_df.show()

tower_logs_region1_df.write.mode("overwrite").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_logs_region1.csv",header=True)




In [0]:
read_tower_logs_region1_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_logs_region1.csv",header=True)
read_tower_logs_region1_df.show()
read_tower_logs_region1_df.printSchema()

In [0]:
text_df1=spark.read.text("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer1.csv")
text_df1.show()

data = text_df1.first()[0].strip()
rows = [row.split(",") for row in data.split()]
columns = ["id", "name", "age", "city", "plan"]
df = spark.createDataFrame(rows, columns)
df.show()



In [0]:

%fs head /Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage1.csv

##2. Filesystem operations
1. Write dbutils.fs code to copy the above datasets into your created Volume folders:
Customer → /Volumes/.../customer/
Usage → /Volumes/.../usage/
Tower (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/

2. Write a command to validate whether files were successfully copied

In [0]:
%fs ls /Volumes/telecom_catalog_assign/landing_zone/landing_vol



## Load strings as it is using dbutils.fs.put

### customer_csv

In [0]:
customer_csv = """101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID"""

dbutils.fs.put(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer1.csv",
    customer_csv,
    True
)

display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer"))

### usage_tsv

In [0]:
usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count 101\t320\t1500\t20 102\t120\t4000\t5 103\t540\t600\t52 104\t45\t200\t2 105\t0\t0\t0 '''
import re

usage_tsv = re.sub(r' (?=\d+\t)', '\n', usage_tsv)

dbutils.fs.put(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage1.csv",
    usage_tsv,
    True
)
display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage"))

In [0]:
tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp 5001|101|TWR01|-80|2025-01-10 10:21:54 5004|104|TWR05|-75|2025-01-10 11:01:12 '''
import re

tower_logs_region1 = re.sub(r' (?=\d+\|)', '\n', tower_logs_region1)

dbutils.fs.put(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_logs_region1_put.csv",
    tower_logs_region1,
    True
)

display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"))

In [0]:
display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol"))

##3. Spark Directory Read Use Cases
1. Read all tower logs using:
Path glob filter (example: *.csv)
Multiple paths input
Recursive lookup

2. Demonstrate these 3 reads separately:
Using pathGlobFilter
Using list of paths in spark.read.csv([path1, path2])
Using .option("recursiveFileLookup","true")

3. Compare the outputs and understand when each should be used.

In [0]:
all_df=spark.read.format("csv").option("header","True").option("inferSchema","True").option("recursiveFileLookup","True").load(["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"])
all_df.show()
all_df.printSchema()


In [0]:
all_df=spark.read.format("csv").option("header","True").option("inferSchema","True").option("recursiveFileLookup","True").load(["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.csv","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_logs_region1.csv"])
all_df.show()
all_df.printSchema()



In [0]:
read_cust_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",header=True,inferSchema=True)
read_cust_df.show()
read_cust_df.printSchema()


##4. Schema Inference, Header, and Separator
1. Try the Customer, Usage files with the option and options using read.csv and format function:<br>
header=false, inferSchema=false<br>
or<br>
header=true, inferSchema=true<br>
2. Write a note on What changed when we use header or inferSchema  with true/false?<br>
3. How schema inference handled “abc” in age?<br>


### Header and InferSchema functions difference

header=True,inferSchema=True - if the df have proper header and structure, it is displaying properly

header=False,inferSchema=False - if the df have proper header and structure, with this line, it is taking column as a row for each and every row

In [0]:
#with header=false, inferSchema=false
read_cust_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer1.csv")
read_cust_df.show()
read_cust_df.printSchema()

read_usage_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage1.tsv/")
read_usage_df.show()
read_usage_df.printSchema()


#with header=True, inferSchema=True
read_cust_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",header=True,inferSchema=True)
read_cust_df.show()
read_cust_df.printSchema()

read_usage_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.csv",header=True,inferSchema=True)
read_usage_df.show()
read_usage_df.printSchema()

##5. Column Renaming Usecases
1. Apply column names using string using toDF function for customer data
2. Apply column names and datatype using the schema function for usage data
3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data 

In [0]:
#Apply column names using string using toDF function for customer data

read_cust_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer1.csv").toDF("id","name","age","city","plan")
read_cust_df.show()

In [0]:
#Apply column names and datatype using the schema function for usage data

schema="custid int,call_voice_mins int,data int,sms int"
read_usage_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage1.csv/",sep="\t",header=True,schema=schema)
read_usage_df.show()
read_usage_df.printSchema()

In [0]:
#Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data

from pyspark.sql.types import StructType,StructField,IntegerType,TimestampType
schema_structure=StructType([
  StructField("Event_ID",IntegerType(),True),
  StructField("Cust_ID",IntegerType(),True), 
  StructField("Tower_ID",IntegerType(),True),
  StructField("Sig_Strength",IntegerType(),True),
  StructField("Time_Stamp",TimestampType(),True)
])

read_tower_logs_region1_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_logs_region1_put.csv",header=True,sep="|",schema=schema_structure)
read_tower_logs_region1_df.show(5,False)
read_tower_logs_region1_df.printSchema()

## Spark Write Operations using 
- csv, json, orc, parquet, delta, saveAsTable, insertInto, xml with different write mode, header and sep options

In [0]:
#write cust data into JSON

read_cust_df.show()
read_cust_df.write.mode("overwrite").json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.json")

read_cust_json_df=spark.read.json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.json")
read_cust_json_df.show()

In [0]:
#write usage data into Table
read_usage_df.show()

read_usage_df.write.mode("overwrite").saveAsTable("telecom_catalog_assign.landing_zone.usage_tbl")

read_usage_tbl_df=spark.read.table("telecom_catalog_assign.landing_zone.usage_tbl")
read_usage_tbl_df.show()


##6. Write Operations (Data Conversion/Schema migration) – CSV Format Usecases
1. Write customer data into CSV format using overwrite mode
2. Write usage data into CSV format using append mode
3. Write tower data into CSV format with header enabled and custom separator (|)
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local from the catalog volume location and see the data of any of the above files opening in a notepad++.

##7. Write Operations (Data Conversion/Schema migration)– JSON Format Usecases
1. Write customer data into JSON format using overwrite mode
2. Write usage data into JSON format using append mode and snappy compression format
3. Write tower data into JSON format using ignore mode and observe the behavior of this mode
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##8. Write Operations (Data Conversion/Schema migration) – Parquet Format Usecases
1. Write customer data into Parquet format using overwrite mode and in a gzip format
2. Write usage data into Parquet format using error mode
3. Write tower data into Parquet format with gzip compression option
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##9. Write Operations (Data Conversion/Schema migration) – Orc Format Usecases
1. Write customer data into ORC format using overwrite mode
2. Write usage data into ORC format using append mode
3. Write tower data into ORC format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##10. Write Operations (Data Conversion/Schema migration) – Delta Format Usecases
1. Write customer data into Delta format using overwrite mode
2. Write usage data into Delta format using append mode
3. Write tower data into Delta format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.
6. Compare the parquet location and delta location and try to understand what is the differentiating factor, as both are parquet files only.

##11. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using saveAsTable() as a managed table
2. Write usage data using saveAsTable() with overwrite mode
3. Drop the managed table and verify data removal
4. Go and check the table overview and realize it is in delta format in the Catalog.
5. Use spark.read.sql to write some simple queries on the above tables created.


##12. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using insertInto() in a new table and find the behavior
2. Write usage data using insertTable() with overwrite mode

##13. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data into XML format using rowTag as cust
2. Write usage data into XML format using overwrite mode with the rowTag as usage
3. Download the xml data and open the file in notepad++ and see how the xml file looks like.

##14. Compare all the downloaded files (csv, json, orc, parquet, delta and xml) 
1. Capture the size occupied between all of these file formats and list the formats below based on the order of size from small to big.

###15. Try to do permutation and combination of performing Schema Migration & Data Conversion operations like...
1. Read any one of the above orc data in a dataframe and write it to dbfs in a parquet format
2. Read any one of the above parquet data in a dataframe and write it to dbfs in a delta format
3. Read any one of the above delta data in a dataframe and write it to dbfs in a xml format
4. Read any one of the above delta table in a dataframe and write it to dbfs in a json format
5. Read any one of the above delta table in a dataframe and write it to another table

##16. Do a final exercise of defining one/two liner of... 
1. When to use/benifits csv
2. When to use/benifits json
3. When to use/benifit orc
4. When to use/benifit parquet
5. When to use/benifit delta
6. When to use/benifit xml
7. When to use/benifit delta tables
